# 12 · Scatter diagnostics

Raw bivariate views: persistence against new establishments per 1,000, against
1990 education, and against log population — each with a simple OLS fit line.
Saves a three-panel figure to `../figures/`. **Manuscript: Appendix C.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
})
for c in ["Index_v2", "pct_hs_1990", "pop_2010", "firms_2022"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Index_v2", "pct_hs_1990", "pop_2010", "firms_2022"])
df = df[df["pop_2010"] > 0]

df["rate_2022_per1k"] = 1000.0 * df["firms_2022"] / df["pop_2010"]
df["log_pop_2010"] = np.log(df["pop_2010"])

def iqr_trim(s, k=3.0):  # de-jitter the display only; fit uses untrimmed data
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return s.clip(q1 - k*iqr, q3 + k*iqr)

def panel(ax, x, y, title, ylabel):
    m = np.isfinite(x) & np.isfinite(y)
    b, a = np.polyfit(x[m], y[m], 1)
    xx = np.linspace(x[m].min(), x[m].max(), 200)
    ax.plot(xx, a + b*xx, lw=2, alpha=0.9)
    ax.scatter(iqr_trim(x[m]), iqr_trim(y[m]), s=8, alpha=0.35)
    ax.set_title(title, fontsize=11); ax.set_xlabel("Persistence index", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10); ax.grid(True, alpha=0.25)

fig, ax = plt.subplots(1, 3, figsize=(12, 4), dpi=200)
panel(ax[0], df["Index_v2"], df["rate_2022_per1k"], "Persistence vs new establishments", "New establishments per 1,000")
panel(ax[1], df["Index_v2"], df["pct_hs_1990"], "Persistence vs education (1990)", "% high school or more (1990)")
panel(ax[2], df["Index_v2"], df["log_pop_2010"], "Persistence vs population", "log(population 2010)")
plt.tight_layout()
plt.savefig("../figures/appendix_scatter.png", bbox_inches="tight")
plt.savefig("../figures/appendix_scatter.pdf", bbox_inches="tight")
plt.show()